In [8]:
import numpy as np
import pandas as pd

In [2]:
ins = pd.read_csv('instruments.csv')

In [7]:
ins[(ins['instrument_type']=='FUT') & (ins['name']=='NATGASMINI')]

,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange
27349,121628679,475112,NATGASMINI26MARFUT,NATGASMINI,0,2026-03-26,0.0,0.1,1,FUT,MCX-FUT,MCX
27350,124791303,487466,NATGASMINI26APRFUT,NATGASMINI,0,2026-04-27,0.0,0.1,1,FUT,MCX-FUT,MCX
27351,125057543,488506,NATGASMINI26MAYFUT,NATGASMINI,0,2026-05-26,0.0,0.1,1,FUT,MCX-FUT,MCX
27352,129092103,504266,NATGASMINI26JUNFUT,NATGASMINI,0,2026-06-25,0.0,0.1,1,FUT,MCX-FUT,MCX
27353,137903623,538686,NATGASMINI26JULFUT,NATGASMINI,0,2026-07-28,0.0,0.1,1,FUT,MCX-FUT,MCX
27354,143743239,561497,NATGASMINI26AUGFUT,NATGASMINI,0,2026-08-26,0.0,0.1,1,FUT,MCX-FUT,MCX
60740,31199756,121874,NATGASMINI26MARFUT,NATGASMINI,0,2026-03-26,0.0,0.1,1,FUT,NCO-FUT,NCO
60741,31166732,121745,NATGASMINI26APRFUT,NATGASMINI,0,2026-04-27,0.0,0.1,1,FUT,NCO-FUT,NCO
60742,31133196,121614,NATGASMINI26MAYFUT,NATGASMINI,0,2026-05-26,0.0,0.1,1,FUT,NCO-FUT,NCO
60743,35429644,138397,NATGASMINI26JUNFUT,NATGASMINI,0,2026-06-25,0.0,0.1,1,FUT,NCO-FUT,NCO


In [ ]:
initial_state = {
    0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0
}

policy = {
    0:[],1:[], 2:[],3:[],4:[],5:[],6:[], 7:[], 8:[], 9:[], 10:["TERMINATING"]
}

k=13
for i in range(0,k):
    tmp_state = {}
    tmp_policy = {}
    for j in range(0,10):
        v1 = initial_state[j]
        x1 = initial_state[j+1]
        if len(policy[j])!=0 and policy[j][-1]=="TERMINATING":
            tmp_policy[j] = policy[j].copy()
            tmp_state[j] = initial_state[j]
            continue
        if j==0:
            x2 = 0 

        else:
            x2 = 1 + initial_state[j-1]

        x3 = max(x1, x2)

        if x3==x1:
            tmp = []
            tmp.append("BUY")
            tmp = tmp + policy[j+1].copy()
            tmp_policy[j] = tmp

        else:
            tmp = []
            tmp.append("SELL")
            tmp = tmp +  policy[j-1].copy()
            tmp_policy[j] = tmp
            
        tmp_state[j] = x3

    tmp_state[10] = 100
    tmp_policy[10] = []
    initial_state = tmp_state.copy()
    policy = tmp_policy.copy()



In [190]:
print(initial_state)
# for i in range(0, 11):
#     print(i, policy[i])

{0: 100, 1: 100, 2: 100, 3: 100, 4: 100, 5: 100, 6: 100, 7: 100, 8: 100, 9: 100, 10: 100}


In [185]:
for i in range(0, 11):
    print(i, policy[i])

0 ['BUY', 'BUY', 'BUY', 'SELL', 'SELL', 'SELL']
1 ['BUY', 'BUY', 'BUY', 'SELL', 'SELL', 'SELL']
2 ['BUY', 'BUY', 'SELL', 'SELL', 'SELL', 'SELL']
3 ['BUY', 'BUY', 'SELL', 'SELL', 'SELL', 'SELL']
4 ['BUY', 'BUY', 'BUY', 'BUY', 'BUY', 'BUY', 'TERMINATING']
5 ['BUY', 'BUY', 'BUY', 'BUY', 'BUY', 'TERMINATING']
6 ['BUY', 'BUY', 'BUY', 'BUY', 'TERMINATING']
7 ['BUY', 'BUY', 'BUY', 'TERMINATING']
8 ['BUY', 'BUY', 'TERMINATING']
9 ['BUY', 'TERMINATING']
10 []


In [157]:
x = [1]
y = [2]
x+=y.copy()
x

[1, 2]

In [158]:
y

[2]

In [152]:
y

[2]

In [200]:
def solve(H):
    V = {s: 0 for s in range(11)}
    V[10] = 0

    # action_table[h][s] = best action at state s with h steps remaining
    action_table = {}

    for h in range(1, H + 1):
        new_V = {}
        actions = {}

        for s in range(11):
            if s == 10:
                new_V[s] = 0
                actions[s] = "TERMINAL"
                continue

            # BUY
            if s == 9:
                buy_val = 100 + V[10]
            else:
                buy_val = 0 + V[s + 1]

            # SELL
            if s == 0:
                sell_val = 0 + V[0]
            else:
                sell_val = 1 + V[s - 1]

            if buy_val >= sell_val:
                new_V[s] = buy_val
                actions[s] = "BUY"
            else:
                new_V[s] = sell_val
                actions[s] = "SELL"

        V = new_V
        action_table[h] = actions

    return V, action_table


def trace_path(action_table, start, H):
    """Simulate what actually happens following the policy"""
    s = start
    total_reward = 0
    path = [s]

    for h in range(H, 0, -1):
        # at step h, h steps remaining
        action = action_table[h][s]

        if action == "TERMINAL":
            break
        elif action == "BUY":
            if s == 9:
                total_reward += 100
                s = 10
            else:
                s = s + 1
        elif action == "SELL":
            if s == 0:
                pass  # nothing happens
            else:
                total_reward += 1
                s = s - 1

        path.append(s)

        if s == 10:
            break

    return path, total_reward


# test
for H in [1, 7, 13]:
    V, action_table = solve(H)
    print(V)
    print(action_table[H])
    # path, reward = trace_path(action_table, start=3, H=H)
    # print(f"\nH={H}")
    # print(f"  V(s=3)   = {V[3]}")
    # print(f"  Path     = {path}")
    # print(f"  Reward   = {reward}")
    # print(f"  Policy   = {[action_table[h][s] for h, s in zip(range(H, 0, -1), path)]}")

{0: 0, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 100, 10: 0}
{0: 'BUY', 1: 'SELL', 2: 'SELL', 3: 'SELL', 4: 'SELL', 5: 'SELL', 6: 'SELL', 7: 'SELL', 8: 'SELL', 9: 'BUY', 10: 'TERMINAL'}
{0: 3, 1: 4, 2: 4, 3: 100, 4: 100, 5: 101, 6: 101, 7: 102, 8: 102, 9: 103, 10: 0}
{0: 'BUY', 1: 'BUY', 2: 'BUY', 3: 'BUY', 4: 'BUY', 5: 'BUY', 6: 'BUY', 7: 'BUY', 8: 'BUY', 9: 'SELL', 10: 'TERMINAL'}
{0: 101, 1: 102, 2: 102, 3: 103, 4: 103, 5: 104, 6: 104, 7: 105, 8: 105, 9: 106, 10: 0}
{0: 'BUY', 1: 'BUY', 2: 'BUY', 3: 'BUY', 4: 'BUY', 5: 'BUY', 6: 'BUY', 7: 'BUY', 8: 'BUY', 9: 'SELL', 10: 'TERMINAL'}


In [198]:
action_table

{1: {0: 'BUY',
  1: 'SELL',
  2: 'SELL',
  3: 'SELL',
  4: 'SELL',
  5: 'SELL',
  6: 'SELL',
  7: 'SELL',
  8: 'SELL',
  9: 'BUY',
  10: 'TERMINAL'}}